# UNSUPERVISED IMAGE LEARNING — NO LABELS NEEDED
### FYP: Autonomous Drone Navigation

This notebook automatically clusters your raw drone images into groups (e.g., trees, buildings, animals, ground) without needing any manual labels.

**Instructions:**
1. Change runtime to **GPU** (Runtime > Change runtime type > T4 GPU).
2. Mount your Google Drive.
3. Ensure your images are in `MyDrive/fyp_drone/images/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install umap-learn -q

In [ ]:
import os
import shutil
import pickle
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# CONFIG
DRIVE_ROOT  = '/content/drive/MyDrive/fyp_drone'
IMAGES_DIR  = os.path.join(DRIVE_ROOT, 'images')
OUTPUT_DIR  = os.path.join(DRIVE_ROOT, 'trained_models', 'unsupervised')

N_CLUSTERS     = 5       # Change this if you want more/fewer groups
N_COMPONENTS   = 128     # PCA dimensions
BATCH_SIZE     = 32      
IMAGE_SIZE     = 224     # ResNet expects 224x224

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
class UnlabelledImageDataset(Dataset):
    EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff'}

    def __init__(self, folder, transform=None):
        self.transform = transform
        self.paths = []

        for root, _, files in os.walk(folder):
            for fname in sorted(files):
                ext = os.path.splitext(fname)[1].lower()
                if ext in self.EXTENSIONS:
                    self.paths.append(os.path.join(root, fname))

        print(f"[DATA] Found {len(self.paths):,} images")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        try:
            img = Image.open(path).convert('RGB')
        except:
            img = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), color=0)
        if self.transform:
            img = self.transform(img)
        return img, path

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
print("[FEATURES] Loading pretrained ResNet50...")
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = torch.nn.Identity()
model.eval()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f"[FEATURES] Using device: {device}")

In [ ]:
dataset = UnlabelledImageDataset(IMAGES_DIR, transform=transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

all_features = []
all_paths = []

print("Starting Feature Extraction...")
with torch.no_grad():
    for i, (images, paths) in enumerate(loader):
        images = images.to(device)
        features = model(images).squeeze().cpu().numpy()
        if features.ndim == 1: features = features.reshape(1, -1)
        all_features.append(features)
        all_paths.extend(paths)
        if i % 10 == 0: print(f"  Processed {len(all_paths)} images...")

features_array = np.vstack(all_features)
print(f"Done! Feature matrix: {features_array.shape}")

In [ ]:
print(f"[PCA] Reducing dimensions to {N_COMPONENTS}...")
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features_array)

pca = PCA(n_components=min(N_COMPONENTS, features_array.shape[0]), random_state=42)
features_reduced = pca.fit_transform(features_scaled)
print(f"[PCA] Variance retained: {pca.explained_variance_ratio_.sum()*100:.1f}%")

In [ ]:
print(f"[CLUSTER] Finding {N_CLUSTERS} groups...")
kmeans = MiniBatchKMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
labels = kmeans.fit_predict(features_reduced)

sil = silhouette_score(features_reduced, labels, sample_size=5000)
print(f"[CLUSTER] Silhouette Score: {sil:.4f}")

In [ ]:
import umap
reducer = umap.UMAP(n_components=2, random_state=42)
embedding = reducer.fit_transform(features_reduced)

plt.figure(figsize=(10, 7))
plt.scatter(embedding[:, 0], embedding[:, 1], c=labels, cmap='tab10', s=5, alpha=0.6)
plt.colorbar(label='Cluster ID')
plt.title('Unsupervised Image Clusters (UMAP Projection)')
plt.show()

In [ ]:
print("[SAVE] Saving models and samples...")
with open(os.path.join(OUTPUT_DIR, 'cluster_model.pkl'), 'wb') as f: pickle.dump(kmeans, f)
with open(os.path.join(OUTPUT_DIR, 'pca_model.pkl'), 'wb') as f: pickle.dump(pca, f)
with open(os.path.join(OUTPUT_DIR, 'scaler.pkl'), 'wb') as f: pickle.dump(scaler, f)

samples_root = os.path.join(OUTPUT_DIR, 'cluster_samples')
if os.path.exists(samples_root): shutil.rmtree(samples_root)
os.makedirs(samples_root, exist_ok=True)

for cid in range(N_CLUSTERS):
    c_dir = os.path.join(samples_root, f'cluster_{cid}')
    os.makedirs(c_dir, exist_ok=True)
    c_paths = [all_paths[i] for i, l in enumerate(labels) if l == cid]
    for p in c_paths[:8]:
        shutil.copy2(p, os.path.join(c_dir, os.path.basename(p)))

print(f"All results saved to {OUTPUT_DIR}")